In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/cs-training.csv", index_col=0)

print("Satır, sütun sayısı:", df.shape)
df.head()


Satır, sütun sayısı: (150000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [2]:
print("Hedef değişken dağılımı (SeriousDlqin2yrs):")
print(df["SeriousDlqin2yrs"].value_counts())
print(df["SeriousDlqin2yrs"].value_counts(normalize=True).round(3))

print("\nEksik değerler:")
print(df.isnull().sum())

Hedef değişken dağılımı (SeriousDlqin2yrs):
SeriousDlqin2yrs
0    139974
1     10026
Name: count, dtype: int64
SeriousDlqin2yrs
0    0.933
1    0.067
Name: proportion, dtype: float64

Eksik değerler:
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64


In [3]:
import os
print("Notebook şu klasörde:", os.getcwd())
print("\nBu klasördekiler:", os.listdir("."))
print("\nÜst klasördekiler:", os.listdir(".."))
print("\ndata klasöründekiler:", os.listdir("../data") if os.path.exists("../data") else "../data yok")

Notebook şu klasörde: C:\Users\ASUDE\Desktop\kredi-risk-analizi\notebooks

Bu klasördekiler: ['.ipynb_checkpoints', '01_eda.ipynb']

Üst klasördekiler: ['.git', '.gitignore', '.ipynb_checkpoints', 'dashboard', 'data', 'desktop.ini', 'notebooks', 'src', 'Untitled.ipynb', 'venv']

data klasöründekiler: ['cs-training.csv']


In [4]:
import shutil
shutil.move("../cs-training.csv", "../data/cs-training.csv")
print(os.listdir("../data"))

FileNotFoundError: [Errno 2] No such file or directory: '../cs-training.csv'

In [ ]:
p = "../data/cs-training.csv"
print("Klasör mü?", os.path.isdir(p))
print("Dosya mı?", os.path.isfile(p))
if os.path.isdir(p):
    print("İçindekiler:", os.listdir(p))
    

In [ ]:
import glob, shutil

klasor = "../data/cs-training.csv"
bulunanlar = glob.glob(klasor + "/**/*.csv", recursive=True)
print("Klasörün içinde bulunan CSV dosyaları:", bulunanlar)

if len(bulunanlar) == 1:
    shutil.move(bulunanlar[0], "../data/gecici.csv")
    shutil.rmtree(klasor)
    os.rename("../data/gecici.csv", "../data/cs-training.csv")
    print("Tamam. data klasörü şimdi:", os.listdir("../data"))
    print("Artık dosya mı?", os.path.isfile("../data/cs-training.csv"))
else:
    print("Beklenmedik durum, hiçbir şey değiştirmedim. Bu çıktıyı bana at.")

In [ ]:
print("Hedef değişken dağılımı (SeriousDlqin2yrs):")
print(df["SeriousDlqin2yrs"].value_counts())
print(df["SeriousDlqin2yrs"].value_counts(normalize=True).round(3))

print("\nEksik değerler:")
print(df.isnull().sum())

In [ ]:
pd.set_option("display.float_format", "{:,.2f}".format)
display(df.describe().T)

print("Yaşı 0 veya 18'den küçük:", (df["age"] < 18).sum())
print("Utilization > 1 olanlar:", (df["RevolvingUtilizationOfUnsecuredLines"] > 1).sum())
print("DebtRatio > 1 olanlar:", (df["DebtRatio"] > 1).sum())
print("\n'Gecikme' sütunlarındaki en yüksek değerler:")
print(df["NumberOfTimes90DaysLate"].value_counts().sort_index().tail(6))

In [ ]:
# 1) 96/98 kodlu satırların temerrüt oranı
kod = df["NumberOfTimes90DaysLate"].isin([96, 98])
print("96/98 satır sayısı:", kod.sum())
print("96/98 olanlarda temerrüt oranı:", round(df.loc[kod, "SeriousDlqin2yrs"].mean(), 3))
print("Diğerlerinde temerrüt oranı:  ", round(df.loc[~kod, "SeriousDlqin2yrs"].mean(), 3))

# 2) DebtRatio > 1 olanlar ile eksik gelir ilişkisi
yuksek = df["DebtRatio"] > 1
print("\nDebtRatio > 1 olup geliri eksik olan:", (yuksek & df["MonthlyIncome"].isnull()).sum(), "/", yuksek.sum())
print("DebtRatio > 1 olup geliri olan:      ", (yuksek & df["MonthlyIncome"].notnull()).sum())

# 3) Uç değerler için yüzdelikler
print("\nUtilization yüzdelikleri:")
print(df["RevolvingUtilizationOfUnsecuredLines"].quantile([0.90, 0.95, 0.99, 0.999]))

In [ ]:
df_clean = df.copy()

# 1) Yaşı 18'den küçük satırı sil
df_clean = df_clean[df_clean["age"] >= 18]

# 2) 96/98 kodlarını bayrağa çevir, sayısal değeri medyanla değiştir
gecikme_sutunlari = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse",
]
df_clean["gecikme_kodu"] = df_clean[gecikme_sutunlari].isin([96, 98]).any(axis=1).astype(int)
for c in gecikme_sutunlari:
    df_clean[c] = df_clean[c].replace([96, 98], np.nan)
    df_clean[c] = df_clean[c].fillna(df_clean[c].median())

# 3) Gelir eksik bayrağı; gelir eksikse DebtRatio'ya güvenme
df_clean["gelir_eksik"] = df_clean["MonthlyIncome"].isnull().astype(int)
df_clean.loc[df_clean["gelir_eksik"] == 1, "DebtRatio"] = np.nan

# 4) Eksikleri doldur
df_clean["MonthlyIncome"] = df_clean["MonthlyIncome"].fillna(df_clean["MonthlyIncome"].median())
df_clean["DebtRatio"] = df_clean["DebtRatio"].fillna(df_clean["DebtRatio"].median())
df_clean["NumberOfDependents"] = df_clean["NumberOfDependents"].fillna(0)

# 5) Uç değerleri %99'luk dilimden sınırla
for c in ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome"]:
    df_clean[c] = df_clean[c].clip(upper=df_clean[c].quantile(0.99))

# Kontrol
print("Yeni boyut:", df_clean.shape)
print("Kalan eksik değer:", df_clean.isnull().sum().sum())
print("gecikme_kodu = 1 satır sayısı:", df_clean["gecikme_kodu"].sum())
print("\nGelir eksik olanlarda temerrüt oranı:", round(df_clean.loc[df_clean["gelir_eksik"]==1, "SeriousDlqin2yrs"].mean(), 3))
print("Geliri olanlarda temerrüt oranı:     ", round(df_clean.loc[df_clean["gelir_eksik"]==0, "SeriousDlqin2yrs"].mean(), 3))
display(df_clean.describe().T[["min", "50%", "max"]])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Yaş grupları
yas = pd.cut(df_clean["age"], bins=[17, 30, 40, 50, 60, 70, 110])
df_clean.groupby(yas, observed=True)["SeriousDlqin2yrs"].mean().plot(kind="bar", ax=axes[0, 0], color="#4C72B0")
axes[0, 0].set_title("Yaş grubuna göre temerrüt oranı")

# Kredi kartı kullanım oranı
util = pd.cut(df_clean["RevolvingUtilizationOfUnsecuredLines"], bins=[-0.01, 0.1, 0.3, 0.6, 0.9, 1.0, 1.2])
df_clean.groupby(util, observed=True)["SeriousDlqin2yrs"].mean().plot(kind="bar", ax=axes[0, 1], color="#DD8452")
axes[0, 1].set_title("Kullanım oranına göre temerrüt oranı")

# 90+ gün gecikme sayısı
gec = df_clean["NumberOfTimes90DaysLate"].clip(upper=5)
df_clean.groupby(gec)["SeriousDlqin2yrs"].mean().plot(kind="bar", ax=axes[1, 0], color="#C44E52")
axes[1, 0].set_title("90+ gün gecikme sayısına göre temerrüt oranı (5+ birleşik)")

# Gelir grupları (çeyreklik)
gelir = pd.qcut(df_clean["MonthlyIncome"], q=5, duplicates="drop")
df_clean.groupby(gelir, observed=True)["SeriousDlqin2yrs"].mean().plot(kind="bar", ax=axes[1, 1], color="#55A868")
axes[1, 1].set_title("Gelir grubuna göre temerrüt oranı")

for ax in axes.flat:
    ax.set_ylabel("Temerrüt oranı")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

print("Hedefle korelasyon (mutlak değere göre):")
print(df_clean.corr()["SeriousDlqin2yrs"].drop("SeriousDlqin2yrs").sort_values(key=abs, ascending=False).round(3))

In [6]:
df_clean.to_csv("../data/cs-training-clean.csv", index=False)

NameError: name 'df_clean' is not defined

In [7]:
%whos DataFrame

Variable   Type         Data/Info
---------------------------------
df         DataFrame    Shape: (150000, 11)


In [8]:
df.isnull().sum()

SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

In [9]:
# Gelir eksikti mi bilgisini ayrı sütun olarak sakla (model için faydalı olabilir)
df["MonthlyIncome_missing"] = df["MonthlyIncome"].isnull().astype(int)

# Eksikleri medyan ile doldur
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(df["NumberOfDependents"].median())

df.isnull().sum()

SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
MonthlyIncome_missing                   0
dtype: int64

In [11]:
df.to_csv("../data/cs-training-clean.csv", index=False)

In [12]:
print(df["age"].min())
print(df["NumberOfTimes90DaysLate"].max())
print(df["RevolvingUtilizationOfUnsecuredLines"].max())

0
98
50708.0


In [13]:
print("Başlangıç:", df.shape)

# 1. Yaşı 18'in altında olan satırları at
df = df[df["age"] >= 18]

# 2. 96/98 kodlu hatalı gecikme satırlarını at
df = df[df["NumberOfTimes90DaysLate"] < 90]

# 3. Uç değerleri %99'luk dilimde sınırla (satır silmeden)
for col in ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]:
    sinir = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=sinir)

print("Sonuç:", df.shape)
print(df["age"].min(), df["NumberOfTimes90DaysLate"].max(), df["RevolvingUtilizationOfUnsecuredLines"].max())

Başlangıç: (150000, 12)
Sonuç: (149730, 12)
21 17 1.093343112149999


In [14]:
df.to_csv("../data/cs-training-clean.csv", index=False)